# AUV-Swarm-FL-RL | Kaggle Execution Hub

Notebook này được thiết kế để chạy trên Kaggle Notebook và thực thi pipeline theo thứ tự tối ưu thời gian:
1) Beta sensitivity (Fig 1, 2, 3)
2) Train PPO-only bootstrap model
3) Scheme comparison / ablation (Fig 4, 5, 6)
4) Train + plot baselines medium (Fig 7 - 100 rounds)
5) Train + plot baselines full (Fig 7 - 1000 rounds)
6) Zip kết quả vào /kaggle/working để tải về

## 1. Clone Source Code và Cài Đặt Môi Trường (Kaggle)
Cập nhật URL repository bên dưới về repo GitHub của bạn trước khi chạy cell.

In [ ]:
import os
import shutil

# TODO: Cập nhật URL repo của bạn
REPO_URL = "https://github.com/ngnam1104/AUV-Swarm-RFL.git"
WORKDIR = "/kaggle/working"
REPO_DIR = os.path.join(WORKDIR, "AUV-Swarm-RFL")

# Xóa bản clone cũ nếu tồn tại
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# Clone repo vào thư mục làm việc của Kaggle
!git clone $REPO_URL $REPO_DIR

# Di chuyển vào repo
os.chdir(REPO_DIR)

# Tạo thư mục log giống pipeline PowerShell
os.makedirs("results/logs", exist_ok=True)

# Cài đặt dependencies
!pip install -r requirements.txt

print("Kaggle environment is ready")
print("Current working directory:", os.getcwd())

In [ ]:
import time
from scripts.eval_beta_sensitivity import build_config
from fl_core.simulator import FLSimulator

print("Khởi tạo FLSimulator để đo thời gian 5 round...")
cfg = build_config(m_value=9, max_rounds=5, eval_interval=1)
fl_sim = FLSimulator(cfg)

total_local_train = 0.0
total_select = 0.0
total_agg = 0.0
total_eval = 0.0

print("Bắt đầu chạy 5 round...")
t0 = time.time()
for rnd in range(1, 6):
    accuracy, active_indices, _, _ = fl_sim.sync_run_step(beta=0.5, rnd=rnd)
    total_local_train += fl_sim.last_timing_stats['local_train_and_grad_sec']
    total_select += fl_sim.last_timing_stats['select_active_sec']
    total_agg += fl_sim.last_timing_stats['aggregate_sec']
    total_eval += fl_sim.last_timing_stats['evaluate_sec']
t1 = time.time()

print("\n--- THỐNG KÊ THỜI GIAN 5 ROUND ---")
print(f"Tổng thời gian train mạng cục bộ ({cfg.M} workers): {total_local_train:.2f}s (Trung bình {total_local_train/5:.2f}s/round)")
print(f"Tổng thời gian chọn Active nodes: {total_select:.2f}s")
print(f"Tổng thời gian tổng hợp (Aggregation): {total_agg:.2f}s (Trung bình {total_agg/5:.2f}s/round)")
print(f"Tổng thời gian đánh giá test set: {total_eval:.2f}s")
print(f"Tổng thời gian thực tế đo được cho 5 vòng lặp: {t1 - t0:.2f}s")


## 2. Bước 1 - Beta Sensitivity (Figure 1, 2, 3)
Chạy thí nghiệm độ nhạy của hệ số beta tương tự bước đầu tiên trong script PowerShell.

In [ ]:
!python -u scripts/eval_beta_sensitivity.py \
  --rounds 1000 \
  --enable-early-stopping \
  --m-values 9 16 25 \
  --beta-start 0.1 \
  --beta-end 0.9 \
  --beta-step 0.1 \
  --out-dir results/beta_sensitivity

## Bước 1.5 - Physical Parameter Sensitivity
Quét các tham số vật lý (fm, pm, fL, pL) để vẽ đồ thị hàm Cost (chữ U).

In [ ]:
!python -u scripts/eval_physical_params.py

## 3. Bước 2 - Train PPO-Only Bootstrap Model
Chỉ huấn luyện PPO để tạo model đầu vào cho thí nghiệm Figure 4, 5, 6 (nhanh hơn train đủ 4 thuật toán).

In [ ]:
!python -u scripts/train_baselines.py \
  --m 9 \
  --max-fl-rounds 200 \
  --episodes 500 \
  --eval-interval 5 \
  --algorithms ppo ddpg greedy random \
  --print-every-steps 10 \
  --out-dir results/fig_7 \
  --step-log-file results/logs/baselines_step_log.txt

## 4. Bước 3 - Plot Figure 7 
Vẽ biểu đồ Figure 7 cho kết quả medium (200 rounds).

In [ ]:
!python -u scripts/plot_fig_7.py \
  --input-dir results/fig_7 \
  --sigma 2.0 \
  --out-path results/fig_7/figure7.png

from IPython.display import Image, display
display(Image("results/fig_7/figure7.png"))

## 5. Bước 4 - Scheme Comparison / Ablation (Figure 4, 5, 6)
Chạy Figure 4, 5, 6 ngay sau khi đã có PPO bootstrap model.

In [ ]:
!python -u scripts/run_fig_4_5_6.py \
  --rounds 200 \
  --m-values 9 \
  --model-path results/fig_7/ppo_baseline_model \
  --lag-threshold 1e4 \
  --out-dir results/eval_schemes

## 6. Bước 5 - và Zip Kết Quả
Vẽ Figure 7 full, sau đó nén toàn bộ thư mục results vào /kaggle/working để tải về.

In [ ]:
ZIP_PATH = "/kaggle/working/experiment_results.zip"
!zip -r $ZIP_PATH results/

print("Zip file saved at:", ZIP_PATH)
print("Open the right panel in Kaggle (Output/Files) to download experiment_results.zip")